# Pipeline de Processamento de Dados: Despesas, PIB e Crimes

Este notebook documenta um pipeline completo de processamento e consolidação de dados públicos municipais, integrando três domínios:
- **Despesas Públicas** (GDV - Gestão de Despesas Variadas)
- **Produto Interno Bruto (PIB)**
- **Dados de Criminalidade**

O objetivo é criar um dataset consolidado para análise de padrões de segurança pública, economia e despesas municipais de 2009-2019.

## Inicialização e Configuração

Importação de bibliotecas necessárias e configuração de logging para rastrear a execução do pipeline.

In [ ]:
import duckdb
import os
import time
import logging
import pandas as pd

# Configuração de logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

## 1. Conversão de Dados PIB (CSV → Parquet com Enriquecimento IBGE)

**Objetivo:** Converter arquivos CSV de PIB municipal para formato Parquet, enriquecendo-os com códigos e informações de municípios do IBGE.

**Etapas:**
- Carrega dados de lookup do IBGE com normalização de nomes de municípios
- Realiza parsing e tipagem de dados do CSV com decimal brasileiro
- Faz join left entre dados do PIB e dados do IBGE por nome de município normalizado
- Salva resultado em Parquet com compressão ZSTD

In [ ]:
def convert_pib_csv_to_parquet_with_ibge(domain='pib', encode='utf-8'):
    
    print("Connecting to in-memory database")

    with duckdb.connect(":memory:") as con:

        for year in [2019]:
            csv_file = f'./{domain}-datasets/{domain}-{year}.csv'
            parquet_file = f'./{domain}-datasets/{domain}-{year}.parquet'
            ibge_file = 'codigos_municipios_regioes.csv'

            if os.path.exists(parquet_file):
                print(f"\n--- {parquet_file} already exists. Skipping conversion for {year}... ---")
                continue

            if os.path.exists(csv_file):
                print(f"\nLoading {csv_file}...")
                start_time = time.time()

                query = rf"""
                    WITH ibge_lookup AS (
                        SELECT 
                            CAST(cod_ibge AS INT) AS cod_ibge,
                            UPPER(TRIM(regexp_replace(strip_accents(REPLACE(municipio, '-', ' ')), '\s+', ' ', 'g'))) AS municipio_norm
                        FROM read_csv(
                            '{ibge_file}', 
                            header=True, 
                            encoding='iso-8859_9-1999',     
                            ignore_errors=True
                        )
                        WHERE length(CAST(cod_ibge AS VARCHAR)) > 4
                    ),
                    pib_data AS (
                        SELECT 
                            {year} AS ano_exercicio,
                            column0 AS ds_municipio,
                            TRY_CAST(column1 AS DOUBLE) AS agropecuaria,
                            TRY_CAST(column2 AS DOUBLE) AS industria,
                            TRY_CAST(column3 AS DOUBLE) AS servicos,
                            TRY_CAST(column4 AS DOUBLE) AS adm_publica,
                            TRY_CAST(column5 AS DOUBLE) AS total_excl_adm,
                            TRY_CAST(column6 AS DOUBLE) AS impostos,
                            TRY_CAST(column7 AS DOUBLE) AS pib_total,
                            TRY_CAST(column8 AS DOUBLE) AS pib_per_capita
                        FROM read_csv(
                            '{csv_file}',
                            delim = ',',             
                            decimal_separator = '.', 
                            skip = 10,
                            header = false,
                            encoding = '{encode}',
                            ignore_errors = true
                        )
                        WHERE column0 IS NOT NULL 
                        AND column0 != ''
                        AND column0 NOT LIKE 'Fonte:%'
                        AND column0 NOT LIKE '(1)%'
                        AND column0 NOT LIKE '(2)%'
                        AND column0 NOT LIKE 'Nota:%'
                        AND column0 != 'ESTADO DE SÃO PAULO'
                    )
                    SELECT 
                        ibge.cod_ibge,
                        p.*
                    FROM pib_data p
                    LEFT JOIN ibge_lookup ibge 
                        ON UPPER(TRIM(regexp_replace(strip_accents(
                            REPLACE(
                                REPLACE(
                                    REPLACE(
                                        REPLACE(p.ds_municipio, '-', ' '), 
                                    'Florínia', 'Florínea'),               
                                'São Luís do', 'São Luiz do'),             
                            '''', '')                                      
                        ), '\s+', ' ', 'g'))) = REPLACE(ibge.municipio_norm, '''', '')
                """

                con.sql(query).write_parquet(parquet_file)
            
                elapsed_time = time.time() - start_time
                parquet_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
                
                print(f"SUCCESS: Saved {parquet_size_mb:.2f} MB to {parquet_file}")
                print(f"Time taken: {elapsed_time:.2f} seconds")

                try:
                    rejects_count = con.sql("SELECT count(*) FROM reject_errors").fetchone()[0]
                    if rejects_count > 0:
                        print(f"WARNING: {rejects_count} rows were skipped due to parsing or encoding errors.")
                except duckdb.CatalogException:
                    pass

            else:
                print(f"\nFile {csv_file} not found. Skipping...")

## 2. Consolidação de Dados por Década e Domínio

**Objetivo:** Consolidar múltiplos arquivos Parquet de diferentes anos (2010-2019) de cada domínio em um único arquivo.

**Etapas:**
- Itera sobre domínios (PIB, Crime)
- Usa wildcard pattern matching para ler múltiplos Parquets de um período
- Aplica union_by_name para garantir compatibilidade de schemas
- Salva consolidado com compressão ZSTD

In [ ]:
def consolidar_decada_por_dominio():

    dominios = ['pib', 'crime']
    
    with duckdb.connect(":memory:") as con:
        
        con.execute("PRAGMA memory_limit='12GB'")
        
        con.execute("PRAGMA temp_directory='./duckdb_tmp'")
        
        for dominio in dominios:
            pasta = f'./{dominio}-datasets/'
            arquivo_final = f'./{dominio}-2010-2019.parquet'
            
            print(f"🚀 Iniciando a consolidação da década para o domínio: {dominio.upper()}...")
            
            query = f"""
                COPY (
                    SELECT * 
                    FROM read_parquet('{pasta}{dominio}-20*.parquet', union_by_name=true)
                ) TO '{arquivo_final}' (FORMAT PARQUET, COMPRESSION 'ZSTD')
            """
            
            try:
                con.execute(query)
                
                linhas = con.sql(f"SELECT count(*) FROM read_parquet('{arquivo_final}')").fetchone()[0]
                print(f"✅ {dominio.upper()} consolidado com sucesso!")
                print(f"📊 Arquivo gerado: {arquivo_final}")
                print(f"📈 Total de linhas na década: {linhas:,}\n".replace(',', '.'))
                
            except Exception as e:
                print(f"❌ Erro ao consolidar {dominio}: {e}\n")

## 3. Análise de Estrutura do Dataset

**Objetivo:** Examinar a estrutura e esquema de um arquivo Parquet, exibindo nomes e tipos de colunas.

**Etapas:**
- Conecta ao DuckDB
- Usa DESCRIBE para extrair metadados das colunas
- Exibe tipo, nome e contagem de linhas

In [ ]:
def describe_dataset_columns():
    con = duckdb.connect()

    con.execute("PRAGMA memory_limit='8GB'")

    files = [
        './gdvDespesasExcel-datasets/gdvDespesasExcel-2010_com_ibge.parquet', 
    ]

    for f in files:
        print(f"\n file: {f}")
        linhas = con.execute(f"SELECT count(*) FROM '{f}'").fetchone()[0]
        print(f"Total de linhas: {linhas:,}")

        schema = con.execute(f"DESCRIBE SELECT * FROM '{f}'").df()
        print(schema[['column_name', 'column_type']].to_string(index=False))

## 4. Cálculo de Cardinalidade de Despesas

**Objetivo:** Calcular o número de valores únicos (cardinalidade real) de colunas categóricas após limpeza.

**Etapas:**
- Normaliza colunas com UPPER(), TRIM()
- Conta valores distintos por categoria (Órgão, Função, Subfunção, etc.)
- Usa COUNT(DISTINCT ...) para cada dimensão

In [ ]:
def cardinalidade_despesas():
    con = duckdb.connect()
    con.execute("PRAGMA memory_limit='8GB'")

    print("Calculando a cardinalidade REAL (Limpa) das colunas categóricas...\n")

    # Utilizando aspas duplas para mapear exatamente os nomes das colunas do seu Parquet
    query = """
        SELECT 
            COUNT(DISTINCT UPPER(TRIM("Órgão"))) AS qtd_orgaos_limpos,
            COUNT(DISTINCT UPPER(TRIM("Função"))) AS qtd_funcoes_limpas,
            COUNT(DISTINCT UPPER(TRIM("Sub Função"))) AS qtd_subfuncoes_limpas,
            COUNT(DISTINCT UPPER(TRIM("Programa"))) AS qtd_programas_limpos,
            COUNT(DISTINCT UPPER(TRIM("Ação"))) AS qtd_acoes_limpas,
            COUNT(DISTINCT UPPER(TRIM("Funcional Programática"))) AS qtd_func_programatica_limpas,
            COUNT(DISTINCT UPPER(TRIM("Município"))) AS qtd_municipios_limpos,
            COUNT(DISTINCT UPPER(TRIM("Despesa"))) AS qtd_despesas_limpas,
            COUNT(DISTINCT UPPER(TRIM("drs"))) AS qtd_drs_limpos,
            COUNT(DISTINCT UPPER(TRIM("r_saude"))) AS qtd_r_saude_limpos
        FROM './gdvDespesasExcel-datasets/gdvDespesasExcel-2010_com_ibge.parquet'
    """

    # Executa a query e converte o resultado para um DataFrame do Pandas
    resultado = con.execute(query).df()
    
    # Imprime transposto (.T) para facilitar a visualização em formato de lista
    print(resultado.T)

## 5. Extração de Dicionário de Dados

**Objetivo:** Extrair e listar valores únicos de colunas categóricas, com filtro especial para segurança pública.

**Etapas:**
- Extrai valores globais (Funções, Órgãos)
- Filtra dados apenas para segurança pública (ILIKE com wildcard)
- Lista valores únicos limpos para cada dimensão categórica
- Exibe contagem de categorias por dimensão

In [ ]:
def extrair_dicionario_dados():
    con = duckdb.connect()
    con.execute("PRAGMA memory_limit='8GB'")

    print("Extraindo dicionário de dados limpo e filtrado...\n")

    def pegar_lista(coluna, filtro_extra=""):
        query = f"""
            SELECT DISTINCT UPPER(TRIM({coluna})) AS categoria_limpa
            FROM './gdvDespesasExcel-datasets/gdvDespesasExcel-2010_com_ibge.parquet' 
            WHERE {coluna} IS NOT NULL {filtro_extra}
            ORDER BY 1
        """
        return con.execute(query).df()['categoria_limpa'].tolist()

    # 1. Funções e Órgãos (Globais, sem filtro)
    funcoes = pegar_lista('"Função"')
    orgaos = pegar_lista('"Órgão"')

    # --- INÍCIO DOS DADOS FILTRADOS POR SEGURANÇA PÚBLICA ---
    trava_seguranca = 'AND ("Função" ILIKE \'%segurança%\' OR "Função" ILIKE \'%seguranca%\')'
    
    subfuncoes_seguranca = pegar_lista('"Sub Função"', trava_seguranca)
    programas_seguranca = pegar_lista('"Programa"', trava_seguranca)
    acoes_seguranca = pegar_lista('"Ação"', trava_seguranca)
    func_programatica_seguranca = pegar_lista('"Funcional Programática"', trava_seguranca)
    municipios_seguranca = pegar_lista('"Município"', trava_seguranca)
    despesas_seguranca = pegar_lista('"Despesa"', trava_seguranca)
    drs_seguranca = pegar_lista('"drs"', trava_seguranca)
    r_saude_seguranca = pegar_lista('"r_saude"', trava_seguranca)


    # --- ÁREA DE IMPRESSÃO ---
    print(f"🔹 FUNÇÕES REAIS ({len(funcoes)}):")
    print(funcoes, "\n")

    print(f"🔹 ÓRGÃOS REAIS ({len(orgaos)}):")
    print(orgaos, "\n")

    print(f"🔹 SUBFUNÇÕES (Segurança Pública) ({len(subfuncoes_seguranca)}):")
    for sf in subfuncoes_seguranca:
        print(f"  - {sf}")
    print("\n")

    print(f"🔹 PROGRAMAS (Segurança Pública) ({len(programas_seguranca)}):")
    print(programas_seguranca, "\n")

    print(f"🔹 AÇÕES (Segurança Pública) ({len(acoes_seguranca)}):")
    print(acoes_seguranca, "\n")

    print(f"🔹 FUNCIONAL PROGRAMÁTICA (Segurança Pública) ({len(func_programatica_seguranca)}):")
    print(func_programatica_seguranca, "\n")
    
    print(f"🔹 DESPESAS (Segurança Pública) ({len(despesas_seguranca)}):")
    print(despesas_seguranca, "\n")

    print(f"🔹 MUNICÍPIOS (Segurança Pública) ({len(municipios_seguranca)}):")
    print(municipios_seguranca, "\n")

    print(f"🔹 REDES DE SAÚDE (Segurança Pública) ({len(r_saude_seguranca)}):")
    print(r_saude_seguranca, "\n")

## 6. Detecção de Cidades Duplicadas

**Objetivo:** Identificar municípios com código IBGE único (cod_ibge) mas múltiplos nomes cadastrados.

**Etapas:**
- Agrupa por cod_ibge
- Conta valores distintos de ds_municipio por grupo
- Filtra apenas grupos com mais de um nome (HAVING COUNT > 1)

In [ ]:
def encontrar_cidade_duplicada():
    con = duckdb.connect()
    
    print("🕵️ Buscando a cidade com dupla identidade no dataset...\n")
    
    query = """
        SELECT 
            cod_ibge, 
            COUNT(DISTINCT ds_municipio) AS qtd_nomes,
            LIST(DISTINCT ds_municipio) AS nomes_utilizados
        FROM './despesas-datasets/despesas-2009-2018.parquet'
        GROUP BY cod_ibge
        HAVING COUNT(DISTINCT ds_municipio) > 1
    """
    
    resultado = con.execute(query).df()
    
    if resultado.empty:
        print("Nenhuma anomalia encontrada.")
    else:
        print("🚨 Encontramos a inconsistência! Veja quem é o culpado:")
        print(resultado)

## 7. Conversão de Despesas GDV para Parquet com Enriquecimento IBGE

**Objetivo:** Converter múltiplos arquivos CSV de despesas (2009-2020) para Parquet e enriquecer com dados do IBGE.

**Etapas:**
- Carrega base IBGE com município e DRS (Departamento Regional de Saúde)
- Define correções manuais para inconsistências de nomes de municípios
- Itera por cada ano de despesas
- Limpa nomes de municípios aplicando correções
- Realiza JOIN INNER com IBGE (usa INNER para garantir dados válidos)
- Salva em Parquet via DuckDB

In [ ]:
def convert_gdv_to_parquet_with_ibge(domain='gdvDespesasExcel'):
    print("Iniciando processo de conversão e enriquecimento com IBGE...")
    
    # 1. Configurando o DuckDB
    con = duckdb.connect()
    con.execute("PRAGMA memory_limit='8GB'")
    
    # 2. Carrega a base do IBGE e define as correções UMA VEZ (fora do loop para ser rápido)
    print("Carregando base de municípios do IBGE...")
    df_ibge = pd.read_csv('codigos_municipios_regioes.csv', sep=';', encoding='iso-8859-1')
    
    correcoes_municipios = {
        'ARCO IRIS': 'ARCO-IRIS',
        'BIRITIBA-MIRIM': 'BIRITIBA MIRIM',
        'BRODOSQUI': 'BRODOWSKI',
        'EMBU': 'EMBU DAS ARTES',
        'EMBU GUACU': 'EMBU-GUACU',
        'IPAUCU': 'IPAUSSU',
        'MOGI-GUACU': 'MOGI GUACU',
        'MOGI-MIRIM': 'MOGI MIRIM',
        'NOVA LUSITANIA': 'NOVA LUZITANIA',
        'PALMEIRA D_OESTE': "PALMEIRA D'OESTE",
        'SALMORAO': 'SALMOURAO',
        'SEVERINEA': 'SEVERINIA',
        'SUD MENUCCI': 'SUD MENNUCCI',
        'SUZANOPOLIS': 'SUZANAPOLIS'
    }

    # 3. Loop pelos anos
    for year in range(2009, 2021):
        csv_file = f'./{domain}-datasets/{domain}-{year}.csv'
        parquet_file = f'./{domain}-datasets/{domain}-{year}_com_ibge.parquet'

        if os.path.exists(parquet_file):
            print(f"\n--- {parquet_file} já existe. Pulando conversão do ano {year}... ---")
            continue

        if os.path.exists(csv_file):
            print(f"\nCarregando e processando {csv_file}...")
            start_time = time.time()

            # Lendo o CSV com Pandas para tipar números brasileiros corretamente
            df = pd.read_csv(
                csv_file, 
                encoding='iso-8859-1', 
                sep=',', 
                low_memory=False,
                decimal=',',
                thousands='.',
                usecols=lambda c: not c.startswith('Unnamed:')
            )
            
            # Limpeza da coluna Município
            if 'Município' in df.columns:
                # Usa .str[-1] para pegar sempre a última parte do split, com ou sem o prefixo
                df['Município'] = df['Município'].astype(str).str.split(' - ', n=1).str[-1].str.strip()
                df['Município'] = df['Município'].replace(correcoes_municipios)
            
            # Cruzamento e Exportação direta via DuckDB
            query_exportacao = f"""
                COPY (
                    SELECT 
                        despesas.*,
                        ibge.cod_ibge,
                        ibge.drs,
                        ibge.r_saude
                    FROM df AS despesas
                    INNER JOIN df_ibge AS ibge 
                        ON UPPER(strip_accents(TRIM(despesas.Município))) = UPPER(strip_accents(TRIM(ibge.municipio)))
                ) TO '{parquet_file}' (FORMAT PARQUET);
            """
            
            con.execute(query_exportacao)
            
            # Cálculos de performance e log de sucesso
            elapsed_time = time.time() - start_time
            parquet_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
            
            print(f"SUCESSO: Salvo {parquet_size_mb:.2f} MB em {parquet_file}")
            print(f"Tempo levado: {elapsed_time:.2f} segundos")

        else:
            print(f"\nArquivo {csv_file} não encontrado. Pulando...")

## 8. Geração do Master Parquet com Agregações SQL

**Objetivo:** Criar um dataset consolidado de despesas com bucketing inteligente por Órgão, Função, Subfunção, Programa, Ação e Tipo de Despesa.

**Etapas Principais:**
1. **CTE dados_brutos**: Carrega todos os Parquets de despesas (2009-2020), normaliza dados
2. **Bucketing por dimensões**: Cria categorias normalizadas para cada dimensão
   - Órgão: seguranca, saude, educacao, maquina, outros
   - Função: idem
   - Subfunção: apenas para segurança (policiamento, defesa_civil, inteligencia)
   - Programa: operacional, infraestrutura, adm_suporte
   - Ação: operacional, inteligencia, infraestrutura, adm_suporte
   - Tipo de Despesa: pessoal, materiais, servicos, investimentos
3. **Agregação Super-Pivot**: Cria colunas de soma e porcentagem para cada bucket

In [ ]:
def gerar_master_parquet_via_sql():
    start_time = time.time()
    print("Iniciando processamento analítico nativo no DuckDB (SQL)...")
    
    con = duckdb.connect()
    con.execute("PRAGMA memory_limit='8GB'")
    
    arquivo_destino = './gdvDespesasExcel-datasets/master_despesas_municipios_2010_2019.parquet'
    os.makedirs(os.path.dirname(arquivo_destino), exist_ok=True)

    query = f"""
        COPY (
            WITH dados_brutos AS (
                SELECT 
                    CAST(regexp_extract(filename, 'gdvDespesasExcel-(\d+)', 1) AS INTEGER) AS Ano,
                    cod_ibge,
                    "Município",
                    (COALESCE("Pago", 0) + COALESCE("Pago Restos", 0)) AS pago_total,
                    
                    -- 1. BUCKET: ÓRGÃO (Administrativo)
                    CASE 
                        WHEN UPPER("Órgão") LIKE '%SEGURANCA%' OR UPPER("Órgão") LIKE '%SEGURANÇA%' THEN 'seguranca'
                        WHEN UPPER("Órgão") LIKE '%SAUDE%' OR UPPER("Órgão") LIKE '%SAÚDE%' THEN 'saude'
                        WHEN UPPER("Órgão") LIKE '%EDUCACAO%' OR UPPER("Órgão") LIKE '%EDUCAÇÃO%' THEN 'educacao'
                        WHEN UPPER("Órgão") LIKE '%ADMINISTRACAO%' OR UPPER("Órgão") LIKE '%JUSTICA%' OR UPPER("Órgão") LIKE '%TRIBUNAL%' THEN 'maquina'
                        ELSE 'outros'
                    END AS b_orgao,

                    -- 2. BUCKET: FUNÇÃO (Área Governamental)
                    CASE 
                        WHEN UPPER("Função") LIKE '%SEGURANCA%' OR UPPER("Função") LIKE '%SEGURANÇA%' THEN 'seguranca'
                        WHEN UPPER("Função") LIKE '%SAUDE%' OR UPPER("Função") LIKE '%SAÚDE%' THEN 'saude'
                        WHEN UPPER("Função") LIKE '%EDUCACAO%' OR UPPER("Função") LIKE '%EDUCAÇÃO%' THEN 'educacao'
                        WHEN UPPER("Função") IN ('01 - LEGISLATIVA', '02 - JUDICIARIA', '03 - ESSENCIAL A JUSTICA', '04 - ADMINISTRACAO') THEN 'maquina'
                        ELSE 'outros'
                    END AS b_funcao,
                    
                    -- 3. BUCKET: TIPO DE DESPESA
                    CASE 
                        WHEN REGEXP_MATCHES(UPPER("Despesa"), 'PESSOAL|ENCARGOS|PROVENTOS') THEN 'pessoal'
                        WHEN REGEXP_MATCHES(UPPER("Despesa"), 'MATERIAL|CONSUMO') THEN 'materiais'
                        WHEN REGEXP_MATCHES(UPPER("Despesa"), 'SERVICO|SERVIÇO') THEN 'servicos'
                        WHEN REGEXP_MATCHES(UPPER("Despesa"), 'INVESTIMENTO|OBRAS|EQUIPAMENTO') THEN 'investimentos'
                        ELSE 'outras'
                    END AS b_despesa
                    
                FROM read_parquet('./gdvDespesasExcel-datasets/gdvDespesasExcel-*_com_ibge.parquet', filename=true)
            )
            
            -- AGREGAÇÃO CONDICIONAL (SUPER PIVOT)
            SELECT 
                Ano,
                cod_ibge,
                "Município" AS city,
                
                -- Pivot: Órgãos
                SUM(CASE WHEN b_orgao = 'seguranca' THEN pago_total ELSE 0 END) AS orgao_seguranca,
                SUM(CASE WHEN b_orgao = 'saude' THEN pago_total ELSE 0 END) AS orgao_saude,
                SUM(CASE WHEN b_orgao = 'educacao' THEN pago_total ELSE 0 END) AS orgao_educacao,
                SUM(CASE WHEN b_orgao = 'maquina' THEN pago_total ELSE 0 END) AS orgao_maquina,
                
                -- Pivot: Funções
                SUM(CASE WHEN b_funcao = 'seguranca' THEN pago_total ELSE 0 END) AS funcao_seguranca,
                SUM(CASE WHEN b_funcao = 'saude' THEN pago_total ELSE 0 END) AS funcao_saude,
                SUM(CASE WHEN b_funcao = 'educacao' THEN pago_total ELSE 0 END) AS funcao_educacao,
                SUM(CASE WHEN b_funcao = 'maquina' THEN pago_total ELSE 0 END) AS funcao_maquina,
                
                -- Porcentagens de Segurança
                (SUM(CASE WHEN b_orgao = 'seguranca' THEN pago_total ELSE 0 END) / NULLIF(SUM(pago_total), 0)) * 100 AS pct_orgao_seguranca,
                (SUM(CASE WHEN b_funcao = 'seguranca' THEN pago_total ELSE 0 END) / NULLIF(SUM(pago_total), 0)) * 100 AS pct_funcao_seguranca,
                
                -- Tipo de Despesa
                SUM(CASE WHEN b_despesa = 'pessoal' THEN pago_total ELSE 0 END) AS desp_pessoal_encargos,
                SUM(CASE WHEN b_despesa = 'materiais' THEN pago_total ELSE 0 END) AS desp_materiais,
                SUM(CASE WHEN b_despesa = 'servicos' THEN pago_total ELSE 0 END) AS desp_servicos_terceiros,
                SUM(CASE WHEN b_despesa = 'investimentos' THEN pago_total ELSE 0 END) AS desp_investimentos_obras
                
            FROM dados_brutos
            GROUP BY Ano, cod_ibge, "Município"
            ORDER BY Ano, cod_ibge
        ) TO '{arquivo_destino}' (FORMAT PARQUET, COMPRESSION 'ZSTD');
    """
    
    con.execute(query)
    
    elapsed_time = time.time() - start_time
    total_linhas = con.execute(f"SELECT count(*) FROM '{arquivo_destino}'").fetchone()[0]
    
    print("\n" + "="*60)
    print("🚀 MASTER PARQUET (COM METRICAS DE RESTO) GERADO! 🚀")
    print("="*60)
    print(f"Linhas consoladas: {total_linhas:,}")
    print(f"Tempo de execução: {elapsed_time:.2f} segundos")
    print("="*60)

## 9. Geração do Dataset Master Final (3 Domínios Consolidados)

**Objetivo:** Consolidar dados de 3 domínios distintos (Despesas, PIB, Crimes) em um único dataset analítico por município e ano.

**Processo:**
1. **CTE annual_crimes**: Agrega dados criminais por ano, região e município
   - Agrupa crimes por tipo: patrimônio, violentos contra vida, dignidade sexual, trânsito, outros
2. **Seleção consolidada**: Seleciona features de 3 domínios
   - **Despesas**: Órgão, Função, Subfunção, Tipo de Despesa (valores e porcentagens)
   - **PIB**: Total, per capita, por setor (agropecuária, indústria, serviços)
   - **Crimes**: Agregados por categoria
3. **JOINs LEFT**: Garante dados mesmo se uma dimensão não tiver registros para um município/ano
4. **Saída**: Parquet comprimido ZSTD com 58 colunas de features

In [ ]:
def gerar_master_dataset_final():
    start_time = time.time()
    print("Iniciando a junção final dos 3 domínios (Despesas, PIB e Crimes) com Features Avançadas...")
    
    con = duckdb.connect()
    con.execute("PRAGMA memory_limit='8GB'")

    # Caminhos baseados na sua infraestrutura de pastas
    arquivo_despesas = './gdvDespesasExcel-datasets/master_despesas_municipios_2010_2019.parquet'
    arquivo_pib = './pib-datasets/pib-2010-2019.parquet' 
    arquivo_crimes = './crime-datasets/crime-2010-2019.parquet' 
    arquivo_destino = './dataset-mestre/master_dataset_2010_2019.parquet'

    # Garante a existência do diretório e limpa arquivos antigos
    os.makedirs(os.path.dirname(arquivo_destino), exist_ok=True)
    if os.path.exists(arquivo_destino):
        os.remove(arquivo_destino)

    query = f"""
        COPY (
            -- 1. CTE: Agrupamento Anual do dataset de crimes mantendo a região geográfica
            WITH annual_crimes AS (
                SELECT 
                    cod_ibge,
                    year,
                    region,
                    SUM(COALESCE(total_de_roubo_outros, 0) + COALESCE(roubo_de_veiculo, 0) + 
                        COALESCE(furto_outros, 0) + COALESCE(furto_de_veiculo, 0)) AS crimes_patrimonio,
                    
                    SUM(COALESCE(homicidio_doloso, 0) + COALESCE(latrocinio, 0) + 
                        COALESCE(lesao_corporal_seguida_de_morte, 0) + COALESCE(tentativa_de_homicidio, 0)) AS crimes_violentos_vida,
                    
                    SUM(COALESCE(total_de_estupro, 0)) AS crimes_dignidade_sexual,
                    
                    SUM(COALESCE(homicidio_culposo_outros, 0) + COALESCE(homicidio_culposo_por_acidente_de_transito, 0) +
                        COALESCE(lesao_corporal_culposa_outras, 0) + COALESCE(lesao_corporal_culposa_por_acidente_de_transito, 0)) AS crimes_transito_e_culposos,
                    
                    SUM(COALESCE(lesao_corporal_dolosa, 0)) AS crimes_outros_violentos
                FROM '{arquivo_crimes}'
                GROUP BY cod_ibge, year, region
            )
            
            -- 2. Seleção Consolidada com todas as features financeiras limpas e separadas
            SELECT 
                -- Chaves Primárias e Identificadores
                d.Ano AS year,
                d.cod_ibge,
                d.city,
                c.region,
                
                -- DESPESAS PÚBLICAS: Valores por Órgão
                d.orgao_seguranca,
                d.orgao_saude,
                d.orgao_educacao,
                d.orgao_maquina,
                
                -- DESPESAS PÚBLICAS: Valores por Função
                d.funcao_seguranca,
                d.funcao_saude,
                d.funcao_educacao,
                d.funcao_maquina,
                
                -- DESPESAS PÚBLICAS: Porcentagens
                d.pct_orgao_seguranca,
                d.pct_funcao_seguranca,
                d.pct_orgao_resto,
                d.pct_funcao_resto,
                
                -- DESPESAS PÚBLICAS: Tipo de Despesa
                d.desp_pessoal_encargos,
                d.desp_materiais,
                d.desp_servicos_terceiros,
                d.desp_investimentos_obras,
                
                -- PIB: Total e Per Capita
                p.pib_total,
                p.pib_per_capita,
                p.agropecuaria AS pib_agropecuaria,
                p.industria AS pib_industria,
                p.servicos AS pib_servicos,
                p.adm_publica AS pib_adm_publica,
                p.impostos AS pib_impostos_liquidos,

                -- CRIMINALIDADE: Agregados por Categoria
                c.crimes_patrimonio,
                c.crimes_violentos_vida,
                c.crimes_dignidade_sexual,
                c.crimes_transito_e_culposos,
                c.crimes_outros_violentos

            FROM '{arquivo_despesas}' AS d
            LEFT JOIN '{arquivo_pib}' AS p 
                ON CAST(d.cod_ibge AS VARCHAR) = CAST(p.cod_ibge AS VARCHAR) 
                AND d.Ano = p.ano_exercicio
            LEFT JOIN annual_crimes AS c 
                ON CAST(d.cod_ibge AS VARCHAR) = CAST(c.cod_ibge AS VARCHAR) 
                AND d.Ano = c.year
                
        ) TO '{arquivo_destino}' (FORMAT PARQUET, COMPRESSION 'ZSTD')
    """
    
    con.execute(query)
    
    elapsed_time = time.time() - start_time
    total_linhas = con.execute(f"SELECT count(*) FROM '{arquivo_destino}'").fetchone()[0]
    
    print("\n" + "="*65)
    print("🚀 DATASET MASTER FINAL CONCLUÍDO E SEPARADO POR HIERARQUIAS! 🚀")
    print("="*65)
    print(f"Total de Linhas Consolidadas: {total_linhas:,}")
    print(f"Tempo de Execução do Super-Join: {elapsed_time:.2f} segundos")
    print(f"Destino Salvo com ZSTD: {arquivo_destino}")
    print("="*65)

## Execução do Pipeline Completo

Executar as funções na ordem apropriada para processar todos os dados:

In [ ]:
if __name__ == "__main__":

    describe_dataset_columns()
    print("------------------------------------", "\n")
    cardinalidade_despesas()
    print("------------------------------------", "\n")
    extrair_dicionario_dados()
    print("------------------------------------", "\n")
    #gerar_master_parquet_via_sql()
    gerar_master_dataset_final()

## Resumo do Pipeline

| Etapa | Função | Entrada | Saída | Descrição |
|-------|--------|---------|-------|----------|
| 1 | `convert_pib_csv_to_parquet_with_ibge` | CSV PIB | Parquet PIB | Converte PIB e enriquece com IBGE |
| 2 | `consolidar_decada_por_dominio` | Parquets por ano | Parquet 2010-2019 | Consolida múltiplos anos em um arquivo |
| 3 | `describe_dataset_columns` | Parquet GDV | Descrição SQL | Examina estrutura do dataset |
| 4 | `cardinalidade_despesas` | Parquet GDV | Estatísticas | Calcula valores únicos por coluna |
| 5 | `extrair_dicionario_dados` | Parquet GDV | Dicionário | Lista valores únicos filtrados |
| 6 | `encontrar_cidade_duplicada` | Parquet Despesas | Anomalias | Identifica municípios com duplos nomes |
| 7 | `convert_gdv_to_parquet_with_ibge` | CSV GDV | Parquet GDV com IBGE | Converte Despesas e enriquece |
| 8 | `gerar_master_parquet_via_sql` | Parquets GDV | Master Parquet | Pivota e agrega por município |
| 9 | `gerar_master_dataset_final` | 3 Parquets | Dataset Final | Consolida 3 domínios em um arquivo |

**Dataset Final:** 58 colunas com Despesas, PIB e Crimes municipais de 2010-2019